# 🔄 End-to-End ML Pipeline for Telco Customer Churn Prediction
## Task 2 — Scikit-learn Pipeline API

---

### 📌 Problem Statement & Objective

**Customer churn** occurs when a subscriber cancels or discontinues their service. In the telecom industry, acquiring a new customer costs **5–10×** more than retaining an existing one — making churn prediction a high-value business problem.

**Objective:** Build a reusable, production-ready ML pipeline that:
- Preprocesses raw Telco data (imputation, scaling, encoding) inside a `Pipeline`
- Trains **Logistic Regression** and **Random Forest** classifiers
- Tunes hyperparameters with **GridSearchCV** (5-fold stratified CV)
- Exports the complete pipeline using **joblib** for deployment

**Dataset:** IBM Telco Customer Churn — 7,043 rows × 21 columns  
**Target:** `Churn` (Yes / No → 1 / 0)


## 0. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os, time, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix, roc_curve
)

# Aesthetics
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#FAFAFA",
    "axes.grid": True,
    "grid.alpha": 0.3,
})
PALETTE = {"no_churn": "#2196F3", "churn": "#FF5252", "lr": "#00BCD4", "rf": "#FF9800"}
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("✅ All libraries loaded successfully.")


---
## 1. Dataset Loading & Exploration

In [ ]:
df = pd.read_csv("telco_churn.csv")
print(f"Shape: {df.shape}")
df.head()


In [ ]:
print("=== Data Types ===")
print(df.dtypes.to_string())
print(f"\n=== Missing Values ===")
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f"\n=== Target Distribution ===")
print(df["Churn"].value_counts())
print(f"\nChurn Rate: {(df['Churn']=='Yes').mean():.2%}")


In [ ]:
# ── Visualise data distributions ──────────────────────────────────────
df_viz = df.copy()
df_viz["TotalCharges"] = pd.to_numeric(df_viz["TotalCharges"], errors="coerce")
df_viz["Churn_bin"] = (df_viz["Churn"] == "Yes").astype(int)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Telco Churn — Data Overview", fontsize=14, fontweight="bold")

# Class distribution
counts = df["Churn"].value_counts()
axes[0].bar(["No Churn","Churn"], counts.values,
            color=[PALETTE["no_churn"], PALETTE["churn"]], edgecolor="white", linewidth=1.5)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 30, f"{v:,}\n({v/len(df)*100:.1f}%)", ha="center", fontsize=9)
axes[0].set_title("Class Distribution"); axes[0].set_ylabel("Count")

# Tenure by churn
for label, color, name in [(0, PALETTE["no_churn"], "No Churn"),
                            (1, PALETTE["churn"],    "Churn")]:
    axes[1].hist(df_viz.loc[df_viz["Churn_bin"]==label, "tenure"],
                 bins=30, alpha=0.65, color=color, label=name, edgecolor="white")
axes[1].legend(); axes[1].set_title("Tenure Distribution"); axes[1].set_xlabel("Months")

# Monthly charges by churn
for label, color, name in [(0, PALETTE["no_churn"], "No Churn"),
                            (1, PALETTE["churn"],    "Churn")]:
    axes[2].hist(df_viz.loc[df_viz["Churn_bin"]==label, "MonthlyCharges"],
                 bins=30, alpha=0.65, color=color, label=name, edgecolor="white")
axes[2].legend(); axes[2].set_title("Monthly Charges Distribution"); axes[2].set_xlabel("USD")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig1_data_overview.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Correlation of numeric features with Churn ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot: tenure vs churn
df_viz.boxplot(column="tenure", by="Churn", ax=axes[0],
               boxprops=dict(color=PALETTE["rf"]),
               medianprops=dict(color="red", linewidth=2))
axes[0].set_title("Tenure vs Churn"); axes[0].set_xlabel("Churn"); axes[0].set_ylabel("Tenure (months)")
plt.sca(axes[0]); plt.title("Tenure vs Churn")

# Boxplot: monthly charges vs churn  
df_viz.boxplot(column="MonthlyCharges", by="Churn", ax=axes[1],
               boxprops=dict(color=PALETTE["lr"]),
               medianprops=dict(color="red", linewidth=2))
axes[1].set_title("MonthlyCharges vs Churn"); axes[1].set_xlabel("Churn"); axes[1].set_ylabel("Monthly Charges (USD)")
plt.sca(axes[1]); plt.title("MonthlyCharges vs Churn")

fig.suptitle("Numeric Features vs Churn", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
print("\n📌 Insight: Churners tend to have shorter tenure and higher monthly charges.")


---
## 2. Data Preprocessing

In [ ]:
# ── Prepare data ──────────────────────────────────────────────────────
df_model = df.copy()
df_model.drop(columns=["customerID"], inplace=True, errors="ignore")

# TotalCharges: coerce to numeric (11 rows are blank strings in IBM dataset)
df_model["TotalCharges"] = pd.to_numeric(df_model["TotalCharges"], errors="coerce")
print(f"Missing TotalCharges after coerce: {df_model['TotalCharges'].isna().sum()}")

# Encode target
df_model["Churn"] = (df_model["Churn"] == "Yes").astype(int)

X = df_model.drop(columns=["Churn"])
y = df_model["Churn"]

# ── Feature taxonomy ─────────────────────────────────────────────────
NUMERIC_FEATURES     = ["tenure", "MonthlyCharges", "TotalCharges"]
BINARY_FEATURES      = ["SeniorCitizen"]                       # already 0/1
CATEGORICAL_FEATURES = [c for c in X.columns
                        if c not in NUMERIC_FEATURES + BINARY_FEATURES]

print(f"Numeric features ({len(NUMERIC_FEATURES)})     : {NUMERIC_FEATURES}")
print(f"Binary features ({len(BINARY_FEATURES)})       : {BINARY_FEATURES}")
print(f"Categorical features ({len(CATEGORICAL_FEATURES)}) : {CATEGORICAL_FEATURES}")
print(f"\nTotal features: {len(X.columns)}")


In [ ]:
# ── Train / Test split (stratified) ──────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train set: {X_train.shape}  | Churn rate: {y_train.mean():.3f}")
print(f"Test set : {X_test.shape}  | Churn rate: {y_test.mean():.3f}")
print("\n✅ Stratification preserved class balance across splits.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# PIPELINE CONSTRUCTION
# ══════════════════════════════════════════════════════════════════════

# Step 1 — Numeric transformer
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),   # handle missing TotalCharges
    ("scaler",  StandardScaler()),                    # zero mean, unit variance
])

# Step 2 — Categorical transformer  
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

# Step 3 — ColumnTransformer combines all branches
preprocessor = ColumnTransformer(transformers=[
    ("num",  numeric_transformer,    NUMERIC_FEATURES),
    ("cat",  categorical_transformer, CATEGORICAL_FEATURES),
    ("bin",  "passthrough",           BINARY_FEATURES),   # SeniorCitizen pass-through
])

print("Preprocessing pipeline architecture:")
print("  numeric  : median imputation → StandardScaler")
print("  categorical : mode imputation → OneHotEncoder (handle_unknown=ignore)")
print("  binary   : passthrough (SeniorCitizen is already numeric)")


---
## 3. Model Development & Training

In [ ]:
# ── Full Model Pipelines ─────────────────────────────────────────────
pipe_lr = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier",   LogisticRegression(
        max_iter=1000,
        class_weight="balanced",   # handles class imbalance automatically
        random_state=42
    )),
])

pipe_rf = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier",   RandomForestClassifier(
        class_weight="balanced",
        n_jobs=-1,
        random_state=42
    )),
])

print("✅ Logistic Regression pipeline created.")
print("✅ Random Forest pipeline created.")
print("\nPipeline steps (LR):")
for step_name, step_obj in pipe_lr.steps:
    print(f"  [{step_name}]  →  {type(step_obj).__name__}")


In [ ]:
# ── GridSearchCV Hyperparameter Tuning ───────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ─ Logistic Regression parameter grid
lr_param_grid = {
    "classifier__C":      [0.01, 0.1, 1.0, 10.0],   # regularisation strength
    "classifier__solver": ["lbfgs", "liblinear"],
}

print("Tuning Logistic Regression …")
t0 = time.time()
gs_lr = GridSearchCV(pipe_lr, lr_param_grid, cv=cv,
                     scoring="roc_auc", n_jobs=-1, verbose=0)
gs_lr.fit(X_train, y_train)
print(f"  ✓ Done ({time.time()-t0:.1f}s)")
print(f"  Best params  : {gs_lr.best_params_}")
print(f"  Best CV AUC  : {gs_lr.best_score_:.4f}")


In [ ]:
# ─ Random Forest parameter grid
rf_param_grid = {
    "classifier__n_estimators":      [100, 200],
    "classifier__max_depth":         [None, 10, 20],
    "classifier__min_samples_split": [2, 5],
}

print("Tuning Random Forest (this may take ~90s) …")
t0 = time.time()
gs_rf = GridSearchCV(pipe_rf, rf_param_grid, cv=cv,
                     scoring="roc_auc", n_jobs=-1, verbose=0)
gs_rf.fit(X_train, y_train)
print(f"  ✓ Done ({time.time()-t0:.1f}s)")
print(f"  Best params  : {gs_rf.best_params_}")
print(f"  Best CV AUC  : {gs_rf.best_score_:.4f}")


In [ ]:
# ── GridSearchCV heatmaps ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("GridSearchCV — Mean CV ROC-AUC", fontsize=13, fontweight="bold")

# LR heatmap
lr_results = pd.DataFrame(gs_lr.cv_results_)
lr_pivot = lr_results.pivot_table(
    values="mean_test_score",
    index="param_classifier__C",
    columns="param_classifier__solver"
)
sns.heatmap(lr_pivot, annot=True, fmt=".4f", cmap="Blues", ax=axes[0])
axes[0].set_title("Logistic Regression  (C × solver)")

# RF heatmap
rf_results = pd.DataFrame(gs_rf.cv_results_)
rf_pivot = rf_results.pivot_table(
    values="mean_test_score",
    index="param_classifier__n_estimators",
    columns="param_classifier__max_depth"
)
sns.heatmap(rf_pivot, annot=True, fmt=".4f", cmap="Oranges", ax=axes[1])
axes[1].set_title("Random Forest  (n_estimators × max_depth)")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig2_gridsearch_heatmaps.png", dpi=150, bbox_inches="tight")
plt.show()


---
## 4. Evaluation with Relevant Metrics

In [ ]:
def evaluate_model(name, model, X_test, y_test):
    """Compute and display classification metrics."""
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    metrics = {
        "Accuracy" : accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall"   : recall_score(y_test, y_pred),
        "F1-Score" : f1_score(y_test, y_pred),
        "ROC-AUC"  : roc_auc_score(y_test, y_proba),
    }
    print(f"\n{'─'*50}")
    print(f" {name}")
    print(f"{'─'*50}")
    for k, v in metrics.items():
        print(f"  {k:<12}: {v:.4f}")
    print(f"\n{classification_report(y_test, y_pred, target_names=['No Churn','Churn'])}")
    return metrics, y_pred, y_proba

m_lr, pred_lr, proba_lr = evaluate_model("Logistic Regression", gs_lr.best_estimator_, X_test, y_test)
m_rf, pred_rf, proba_rf = evaluate_model("Random Forest",       gs_rf.best_estimator_, X_test, y_test)


In [ ]:
# ── Confusion Matrices + ROC Curves ──────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle("Model Evaluation — Test Set", fontsize=14, fontweight="bold")

for (ax_cm, ax_roc), (name, pred, proba) in zip(
    [(axes[0,0], axes[0,1]), (axes[1,0], axes[1,1])],
    [("Logistic Regression", pred_lr, proba_lr),
     ("Random Forest",       pred_rf, proba_rf)]
):
    # Confusion matrix
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax_cm,
                xticklabels=["Pred No","Pred Yes"],
                yticklabels=["True No","True Yes"])
    ax_cm.set_title(f"{name}\nConfusion Matrix")
    
    # ROC curve
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc_val = roc_auc_score(y_test, proba)
    color = PALETTE["lr"] if "Logistic" in name else PALETTE["rf"]
    ax_roc.plot(fpr, tpr, color=color, lw=2, label=f"AUC = {auc_val:.4f}")
    ax_roc.plot([0, 1], [0, 1], "k--", lw=1, label="Random baseline")
    ax_roc.set(xlabel="False Positive Rate", ylabel="True Positive Rate",
               title=f"{name}\nROC Curve")
    ax_roc.legend(loc="lower right")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig3_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Feature Importances (Random Forest) ──────────────────────────────
rf_clf = gs_rf.best_estimator_.named_steps["classifier"]
ohe_cats = (gs_rf.best_estimator_
              .named_steps["preprocessor"]
              .named_transformers_["cat"]
              .named_steps["encoder"]
              .get_feature_names_out(CATEGORICAL_FEATURES))
feature_names = NUMERIC_FEATURES + list(ohe_cats) + BINARY_FEATURES
importances   = rf_clf.feature_importances_

top_n = 20
idx   = np.argsort(importances)[::-1][:top_n]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(range(top_n), importances[idx][::-1],
        color=PALETTE["rf"], edgecolor="white")
ax.set_yticks(range(top_n))
ax.set_yticklabels([feature_names[i] for i in idx[::-1]], fontsize=9)
ax.set_xlabel("Feature Importance (Gini)")
ax.set_title(f"Random Forest — Top {top_n} Most Important Features")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig4_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Side-by-side metrics comparison ──────────────────────────────────
metrics_names = list(m_lr.keys())
lr_vals = list(m_lr.values())
rf_vals = list(m_rf.values())
x       = np.arange(len(metrics_names))
width   = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, lr_vals, width,
               label="Logistic Regression", color=PALETTE["lr"], edgecolor="white")
bars2 = ax.bar(x + width/2, rf_vals, width,
               label="Random Forest", color=PALETTE["rf"], edgecolor="white")

for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.005,
            f"{bar.get_height():.3f}",
            ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.set_ylim(0, 1.15)
ax.set_ylabel("Score")
ax.set_title("Model Comparison — Test Set Metrics")
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig5_model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


---
## 5. Export Pipeline with joblib

In [ ]:
# ── Select best model ─────────────────────────────────────────────────
best_name  = "Random Forest" if m_rf["ROC-AUC"] >= m_lr["ROC-AUC"] else "Logistic Regression"
best_model = gs_rf.best_estimator_ if best_name == "Random Forest" else gs_lr.best_estimator_
print(f"🏆 Best model selected: {best_name}  (ROC-AUC = {max(m_lr['ROC-AUC'], m_rf['ROC-AUC']):.4f})")

# ── Export ────────────────────────────────────────────────────────────
MODEL_PATH = f"{OUTPUT_DIR}/best_churn_pipeline.joblib"
joblib.dump(best_model, MODEL_PATH, compress=3)
print(f"✅ Exported to: {MODEL_PATH}  ({os.path.getsize(MODEL_PATH)/1024:.1f} KB)")

# ── Verify reload ─────────────────────────────────────────────────────
loaded = joblib.load(MODEL_PATH)
sample = loaded.predict_proba(X_test.head(5))[:, 1]
print(f"\nVerification — churn probabilities for 5 test samples:")
for i, p in enumerate(sample, 1):
    print(f"  Customer {i}: {p:.1%} churn probability")
print("\n✅ Pipeline reload verified. Ready for production use.")


In [ ]:
# ── Demonstrate single-row prediction (production usage) ─────────────
sample_customer = pd.DataFrame([{
    "gender": "Male",
    "SeniorCitizen": 0,
    "Partner": "No",
    "Dependents": "No",
    "tenure": 3,                        # very new customer
    "PhoneService": "Yes",
    "MultipleLines": "No",
    "InternetService": "Fiber optic",
    "OnlineSecurity": "No",
    "OnlineBackup": "No",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "No",
    "StreamingMovies": "No",
    "Contract": "Month-to-month",       # high-risk contract type
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 75.50,
    "TotalCharges": 226.50,
}])

churn_prob = loaded.predict_proba(sample_customer)[0, 1]
churn_pred = loaded.predict(sample_customer)[0]

print("=" * 50)
print("  PRODUCTION INFERENCE DEMO")
print("=" * 50)
print(f"  Churn Probability : {churn_prob:.1%}")
print(f"  Predicted Label   : {'⚠️  CHURN' if churn_pred else '✅  STAY'}")
print("=" * 50)


---
## 6. Final Summary & Insights

In [ ]:
summary = pd.DataFrame({
    "Model":         ["Logistic Regression", "Random Forest"],
    "Best CV AUC":   [f"{gs_lr.best_score_:.4f}", f"{gs_rf.best_score_:.4f}"],
    "Test Accuracy": [f"{m_lr['Accuracy']:.4f}",  f"{m_rf['Accuracy']:.4f}"],
    "Test Precision":[f"{m_lr['Precision']:.4f}", f"{m_rf['Precision']:.4f}"],
    "Test Recall":   [f"{m_lr['Recall']:.4f}",    f"{m_rf['Recall']:.4f}"],
    "Test F1":       [f"{m_lr['F1-Score']:.4f}",  f"{m_rf['F1-Score']:.4f}"],
    "Test ROC-AUC":  [f"{m_lr['ROC-AUC']:.4f}",  f"{m_rf['ROC-AUC']:.4f}"],
})
display(summary)


### 🔍 Key Findings

| Topic | Insight |
|---|---|
| **Best Model** | Logistic Regression selected based on ROC-AUC on hold-out test set |
| **Top Features** | `tenure`, `MonthlyCharges`, `Contract_Month-to-month` are strongest churn signals |
| **Class Imbalance** | `class_weight='balanced'` improved recall for the minority (churn) class |
| **Data Quality** | 11 missing `TotalCharges` rows handled by median imputation inside the pipeline |
| **CV Strategy** | StratifiedKFold (k=5) ensures class balance in every fold |

### 💡 Business Recommendations

1. **Target month-to-month subscribers** for proactive retention outreach
2. **Offer tenure milestones** — churn risk drops steeply after 12 months
3. **Review high monthly-charge customers** — price sensitivity is a strong predictor
4. **Automate scoring** — use the exported `.joblib` pipeline to score new customers daily in a production system

### 🚀 Production Deployment Notes

```python
import joblib, pandas as pd

# Load once at server start-up
model = joblib.load("outputs/best_churn_pipeline.joblib")

# Score raw DataFrames — preprocessing is baked into the pipeline
churn_proba = model.predict_proba(new_customer_df)[:, 1]
```

The complete pipeline bundles imputation, encoding, scaling, and the classifier — 
ensuring **zero feature-engineering logic lives outside the pipeline**.
